# Tạo embedding BGE-M3 cho một pack ZIP

Notebook chỉ clone dự án và gọi mã trong `src/`. Chọn **Runtime > Change runtime type > GPU** khi chạy mô hình. Chạy các ô từ trên xuống.

**Trước khi chạy:** mã nguồn mới phải có trên GitHub; đặt `GIT_REF` đúng nhánh/tag/commit. Notebook không lấy được các thay đổi chỉ nằm trên máy cá nhân. Không đặt token GitHub trong URL hoặc lưu trong notebook. Dự án riêng tư cần cấu hình xác thực Git của phiên Colab trước.

## 1. Clone dự án
Nếu đổi phiên bản mã sau khi đã import module, khởi động lại phiên Python trước khi tiếp tục.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = "https://github.com/qtamtensor05/ViGovBot.git"
GIT_REF = "codex/feat-multi-embedding-colab"  # Nhánh hiện tại; đổi main sau khi merge hoặc dùng commit cố định.
REPO_DIR = Path("/content/ViGovBot")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("REPO_DIR đang trỏ tới dự án khác; chọn thư mục mới")
    changes = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
    if changes.strip():
        raise RuntimeError("Có thay đổi trong checkout Colab; lưu lại hoặc chọn REPO_DIR mới trước khi cập nhật")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Commit đang chạy:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. Cài thư viện
Cài từ file requirements của đúng phiên bản dự án vừa clone.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-embedding.txt"], check=True)

## 3. Kết nối Google Drive
Chọn tài khoản chứa dữ liệu và cấp quyền khi Colab yêu cầu.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Chọn pack và chạy
Đặt các pack ZIP trong `MyDrive/RAG_Data`. Mỗi phiên xử lý một pack khác nhau. Đổi `PACK_ID` và chạy lại ô này cho pack tiếp theo. Prefix đã có sẽ không bị lặp; khi thiếu VRAM, mã worker tự giảm batch và thử lại.

In [ ]:
PACK_ID = "pack_01"
DATA_DIR = Path("/content/drive/MyDrive/RAG_Data")
OUTPUT_DIR = DATA_DIR / "completed"
BATCH_SIZE = 32
MODEL_REVISION = None
command = [sys.executable, "-m", "src.embeddings.pack_worker", "--pack-id", PACK_ID,
           "--zip-path", str(DATA_DIR / f"{PACK_ID}.zip"), "--output-dir", str(OUTPUT_DIR),
           "--batch-size", str(BATCH_SIZE), "--device", "cuda", "--no-mount"]
if MODEL_REVISION:
    command += ["--revision", MODEL_REVISION]
subprocess.run(command, cwd=REPO_DIR, check=True)

Sau khi đủ các cặp vector/metadata, chạy notebook `merge_vector_packs.ipynb`. File đã có sẽ không bị ghi đè; chuyển kết quả chưa hoàn chỉnh sang nơi khác trước khi chạy lại.